# Reading model — replace the word-counter

The word-counter in this project scores **59.76%** when it is examined properly: taught on the ISOT collection, then shown McIntire articles it has never seen. Worse than the number suggests, it labels only **29.5% of real articles correctly** — it shouts FAKE at almost everything.

The reason is that it counts words and ignores their order, so it cannot tell these apart:

> Scientists have proven this cures cancer  
> Scientists have **not** proven this cures cancer

This notebook trains DistilBERT on the same data with the same honest split, so the number it produces is directly comparable to 59.76%.

---

## Before you start

**1. Turn the GPU on.** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save. Without it this takes hours instead of minutes.

**2. Get five files into Google Drive.** Put them in one folder at the top level of your Drive. **Whatever you call that folder, `DRIVE_FOLDER` in section 3 must match it exactly** (this notebook says `fake_news_data`; your Drive currently uses `fakedata`).

The files and put these in it:

| File | Where it is |
|---|---|
| `Fake.csv` | your project folder |
| `True.csv` | your project folder |
| `fake_or_real_news.csv` | your project folder |
| `text_cleaning.py` | your project's `analyzer/` folder |
| `diverse_fakes.csv` | your project folder, built by `fetch_diverse_fakes.py` |

Drive rather than the upload button because Colab disconnects if you leave it idle, and you do not want to upload 110MB twice.

`text_cleaning.py` comes along so the fingerprint-stripping here is *exactly* the rule your Django app uses. Copying the rules into this notebook by hand would let the two drift apart, and then the model gets fed text it was never trained on.

**3. Run the cells top to bottom.** Leave `TRIAL_RUN` switched on for the first pass — it finishes in about four minutes and proves the whole thing works before you commit an hour.

---

## One cost to know about up front

Running this model in your Django app afterwards needs **PyTorch installed locally — about 2.5GB**. Answering is fast (well under a second per article on a normal laptop, no GPU needed), but the install is not small. Worth knowing before you decide to deploy it.

## 1. Check the GPU is actually on

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU, then run this again.")
    print("Training on the processor alone would take hours.")

## 2. Install the reading-model library

Colab already has PyTorch and scikit-learn. `transformers` is the library that fetches DistilBERT and handles the training loop.

In [ ]:
!pip install -q "transformers>=4.40" "accelerate>=0.30"

import transformers
print(f"transformers {transformers.__version__}")

## 3. Fetch the five files from Drive

Google will ask you to authorise access. If a file is missing this cell says which one rather than failing later with something cryptic.

In [ ]:
import shutil
from pathlib import Path

DRIVE_FOLDER = '/content/drive/MyDrive/fake_news_data'
NEEDED = ['Fake.csv', 'True.csv', 'fake_or_real_news.csv', 'text_cleaning.py',
          'diverse_fakes.csv']

from google.colab import drive
drive.mount('/content/drive')

missing = []
for name in NEEDED:
    source = Path(DRIVE_FOLDER) / name
    if source.exists():
        shutil.copy(source, Path('/content') / name)
        size = source.stat().st_size / 1_000_000
        print(f"  got {name}  ({size:.1f} MB)")
    else:
        missing.append(name)

if missing:
    raise SystemExit(
        f"\nMissing from {DRIVE_FOLDER}:\n  "
        + "\n  ".join(missing)
        + "\n\nPut them in that Drive folder and run this cell again."
    )

print("\nAll five files ready.")

## 4. Load the articles and strip the fingerprints

Same preparation as `train_honest_model.py`, so the comparison is fair:

- headline and body joined, exactly how the Django app builds its input
- McIntire articles that also appear in ISOT are dropped, or the exam is not really unseen
- publisher fingerprints removed using the project's own rule

The cleaning barely helped the word-counter, but it matters more here. A reading model would seize on `(Reuters)` even faster than a word-counter does, because it is a far better pattern-spotter. Leaving the markers in would hand it the answer.

In [ ]:
import re
import unicodedata

import pandas as pd

from text_cleaning import clean_article_text


# The headline tidier, needed only to spot articles carried by both collections.
# Kept minimal here rather than uploading a fifth file — it is used for nothing
# but duplicate detection inside this notebook.
def simple_key(headline):
    text = unicodedata.normalize('NFKD', str(headline or '')).lower()
    text = re.sub(r"'", '', text)
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


frames = []
for path, label in [('Fake.csv', 'FAKE'), ('True.csv', 'REAL')]:
    part = pd.read_csv(path)[['title', 'text']].copy()
    part['label'] = label
    part['collection'] = 'ISOT'
    frames.append(part)

part = pd.read_csv('fake_or_real_news.csv')[['title', 'text', 'label']].copy()
part['label'] = part['label'].astype(str).str.strip().str.upper()
part['collection'] = 'McIntire'
frames.append(part)

# NEW: 7,257 fake articles from 150 different websites, built by
# fetch_diverse_fakes.py. Given its own collection name rather than being
# folded into 'ISOT' for two reasons: the counts printed below stay
# readable, and the exam screening further down can check it like ISOT.
part = pd.read_csv('diverse_fakes.csv')[['title', 'text', 'label']].copy()
part['label'] = part['label'].astype(str).str.strip().str.upper()
part['collection'] = 'diverse'
frames.append(part)

df = pd.concat(frames, ignore_index=True).dropna(subset=['text'])
df['title'] = df['title'].fillna('').astype(str)
df['text'] = df['text'].astype(str)
df['raw'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['raw'].str.len() > 0]
df['label_binary'] = (df['label'] == 'FAKE').astype(int)

# Drop McIntire rows that ISOT already contains
keys = df['title'].map(simple_key)
# Screen against EVERY pile the model learns from, not just ISOT. Miss the
# new one and the exam contains articles it studied, which quietly inflates
# the score. fetch_diverse_fakes.py already blocked McIntire headlines when
# it built the file, using the app's stricter tidier; this is a second,
# independent check with simple_key, which costs nothing to run.
taught = set(keys[df['collection'].isin(['ISOT', 'diverse'])]) - {''}
overlap = (df['collection'] == 'McIntire') & keys.isin(taught)
print(f"dropped {int(overlap.sum()):,} McIntire articles that also appear in a training pile")
df = df[~overlap].reset_index(drop=True)

print("stripping fingerprints ...")
df['cleaned'] = df['raw'].map(clean_article_text)
df = df[df['cleaned'].str.len() > 0].reset_index(drop=True)

for collection in ('ISOT', 'diverse', 'McIntire'):
    rows = df[df['collection'] == collection]
    fake = int((rows['label_binary'] == 1).sum())
    print(f"  {collection:9} {len(rows):>7,}  ({fake:,} fake / {len(rows) - fake:,} real)")

## 5. The honest split

Learn from **ISOT only**. Examine on **McIntire only** — a different collection, different outlets, never seen during training. Identical to the split that gave the word-counter 59.76%.

**Leave `TRIAL_RUN = True` for your first pass.** It uses 4,000 articles and finishes in a few minutes. The number it produces is not worth reporting — it is there to prove the notebook runs before you spend an hour.

In [ ]:
TRIAL_RUN = True     # False for the real run
TRIAL_SIZE = 4000

# How much of each article the model reads. 256 word-pieces is roughly the
# headline plus the opening paragraphs — where a fabricated story usually gives
# itself away. 512 reads more and takes twice as long; worth trying later.
MAX_LENGTH = 256

train_df = df[df['collection'].isin(['ISOT', 'diverse'])]
test_df = df[df['collection'] == 'McIntire']

if TRIAL_RUN:
    # An equal number of fake and real, so a trial cannot look good simply by
    # guessing whichever label happens to dominate the sample.
    train_df = pd.concat([
        group.sample(min(len(group), TRIAL_SIZE // 2), random_state=42)
        for _, group in train_df.groupby('label_binary')
    ])
    print(f"*** TRIAL RUN — {len(train_df):,} articles. The score below is NOT the")
    print(f"*** real result. Set TRIAL_RUN = False once this works end to end.\n")

train_texts = train_df['cleaned'].tolist()
train_labels = train_df['label_binary'].tolist()
test_texts = test_df['cleaned'].tolist()
test_labels = test_df['label_binary'].tolist()

print(f"learning from : {len(train_texts):,} articles (ISOT + new websites)")
print(f"examined on   : {len(test_texts):,} McIntire articles (never seen)")

## 6. Turn the text into something the model reads

DistilBERT splits text into word-pieces it already knows. This is not the word-counting from before — the order is kept, which is the entire point.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("tokenising (a minute or two on the full run) ...")
train_enc = tokenizer(train_texts, truncation=True, padding='max_length',
                      max_length=MAX_LENGTH)
test_enc = tokenizer(test_texts, truncation=True, padding='max_length',
                     max_length=MAX_LENGTH)


class ArticleDataset(torch.utils.data.Dataset):
    """Hands the trainer one article at a time."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item


train_ds = ArticleDataset(train_enc, train_labels)
test_ds = ArticleDataset(test_enc, test_labels)
print("done")

## 7. Train

DistilBERT arrives already understanding English. These two passes only teach it this one job on top of that, which is why it needs so little time.

Trial run: about 4 minutes. Full run: roughly 30–45 minutes on a T4. **Keep the tab open** — Colab disconnects idle sessions and you would lose the progress.

In [ ]:
import inspect

import transformers
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments)

# Same convention as the Django app: 0 = REAL, 1 = FAKE
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1},
)

# What we would LIKE to ask for.
#
# TrainingArguments has renamed and removed keyword arguments repeatedly across
# transformers versions, and Colab installs whatever is current on the day you
# run this. Passing a keyword this version has never heard of kills the cell
# with a plain TypeError -- 'warmup_ratio' did exactly that. So instead of
# pinning a version and hoping, ask the installed class what it actually
# accepts and quietly drop anything it does not.
#
# Everything dropped this way is a tuning nicety, not a requirement. Training
# still runs; it just runs without that one refinement.
wanted = {
    'output_dir': '/content/checkpoints',
    'num_train_epochs': 2,
    'per_device_train_batch_size': 32,
    'per_device_eval_batch_size': 64,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'fp16': torch.cuda.is_available(),   # half precision: same result, twice the speed
    'logging_steps': 50,
    'save_strategy': 'no',               # no checkpoints; only the finished model
    'report_to': [],                     # skip the experiment-tracking prompts
}

accepted = set(inspect.signature(TrainingArguments).parameters)
supported = {k: v for k, v in wanted.items() if k in accepted}

dropped = sorted(set(wanted) - set(supported))
if dropped:
    print(f"this transformers ({transformers.__version__}) does not accept:")
    for name in dropped:
        print(f"    {name}  -- skipped")
else:
    print(f"transformers {transformers.__version__}: all settings accepted")

args = TrainingArguments(**supported)

trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()
print("\ntrained")

## 8. The exam

The number to watch is not the accuracy — it is **recall for REAL**. The word-counter managed 0.295, meaning it wrongly flagged 7 real articles in every 10. If that figure has not climbed a long way, the model is still just shouting FAKE at everything and the accuracy is hollow.

In [ ]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# The old model's sorting score, computed from the scores it left behind in
# analyzer/reading_model/mcintire_scores.csv. This is the number to beat.
OLD_SORTING = 74.44
OLD_ACCURACY, OLD_REAL, OLD_FAKE = 67.13, 0.818, 0.524

output = trainer.predict(test_ds)
logits = np.asarray(output.predictions)
predictions = logits.argmax(axis=1)

# Probability of FAKE for each article, needed for the sorting score below.
fake_score = softmax(logits, axis=1)[:, 1]

accuracy = accuracy_score(test_labels, predictions) * 100
report = classification_report(test_labels, predictions,
                               target_names=['REAL', 'FAKE'],
                               digits=3, output_dict=True)

# THE HEADLINE NUMBER.
#
# Recall and accuracy both depend on where the line between FAKE and REAL is
# drawn, and this cell draws it at the default 0.5 -- which is NOT where the
# Django app draws it. The app reads a tuned cutoff out of
# analyzer/reading_model/cutoff.json (currently 0.1).
#
# So a model can look completely different here purely because its lean moved,
# without having got any better or worse at telling the two apart. The sorting
# score has no line in it: hand it one real and one fake article, how often does
# it rate the fake as the faker? That is the honest measure of whether the model
# learned more, and it is what to compare against the old one.
sorting = roc_auc_score(test_labels, fake_score) * 100

print('=' * 68)
print('  TRIAL RUN - not the real result' if TRIAL_RUN
      else '  RESULT - taught on ISOT + new websites, examined on McIntire')
print('=' * 68)
print(classification_report(test_labels, predictions,
                            target_names=['REAL', 'FAKE'], digits=3))

print(f"  SORTING SCORE   {OLD_SORTING:.2f}%  ->  {sorting:.2f}%   "
      f"({sorting - OLD_SORTING:+.2f})")
print('    ^^ the one that matters. No cutoff in it, so it cannot be')
print('       flattered by the model simply leaning a different way.')
print()
print('  Everything below is read at the DEFAULT line, which the app does not')
print('  use. Useful for spotting a wild lean, not for judging the model.')
print(f"    accuracy      {OLD_ACCURACY:.2f}%  ->  {accuracy:.2f}%")
print(f"    REAL recall    {OLD_REAL:.3f}  ->  {report['REAL']['recall']:.3f}")
print(f"    FAKE recall    {OLD_FAKE:.3f}  ->  {report['FAKE']['recall']:.3f}")
print()

gain = sorting - OLD_SORTING
if TRIAL_RUN:
    print(f'  This is 4,000 articles out of ~52,000. If the sorting score is already')
    print(f'  near or above {OLD_SORTING:.1f}%, the full run should beat it comfortably.')
    print('  Set TRIAL_RUN = False and run sections 5 to 8 again for the real number.')
elif gain < 0:
    print(f'  WORSE at sorting ({gain:+.2f}). The new websites hurt. Do not deploy;')
    print('  keep the model currently in analyzer/reading_model/.')
elif gain < 1:
    print(f'  No real gain ({gain:+.2f}). Any change in the recalls above is the model')
    print('  leaning differently, not knowing more. Not worth deploying.')
else:
    print(f'  GENUINELY BETTER at sorting ({gain:+.2f} points).')
    print('  Next step is local, not here: unzip into analyzer/reading_model/ and')
    print('  run tune_cutoff.py to re-pick the line. That is what turns a better')
    print('  sorting score into better verdicts in the app.')


## 9. Save it and bring it home

Skip this on the trial run. Saves to Drive first, because the browser download of a 260MB file sometimes fails and re-running the training to recover it would be annoying.

In [ ]:
import json
from datetime import date

if TRIAL_RUN:
    raise SystemExit('Trial run — nothing saved. Set TRIAL_RUN = False first.')

OUT = Path('/content/reading_model')
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

# The measurements travel with the model, same idea as analyzer/model_meta.json,
# so nothing has to be typed into the app by hand.
metadata = {
    'model_type': 'distilbert',
    'base_model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'test_accuracy': round(accuracy, 2),
    'real_recall': round(report['REAL']['recall'], 3),
    'fake_recall': round(report['FAKE']['recall'], 3),
    'training_samples': len(train_texts),
    'test_samples': len(test_texts),
    'trained_at': date.today().isoformat(),
    'evaluation': 'cross_source',
    'trained_on': 'Kaggle ISOT (Fake.csv + True.csv) + diverse_fakes.csv',
    'diverse_fakes_articles': int((train_df['collection'] == 'diverse').sum()),
    'tested_on': 'McIntire (fake_or_real_news.csv)',
    'text_cleaned': True,
}
(OUT / 'model_meta.json').write_text(json.dumps(metadata, indent=2))

shutil.make_archive('/content/reading_model', 'zip', OUT)
size = Path('/content/reading_model.zip').stat().st_size / 1_000_000
print(f"reading_model.zip  ({size:.0f} MB)")

# Drive copy first — the reliable one
shutil.copy('/content/reading_model.zip', f'{DRIVE_FOLDER}/reading_model.zip')
print(f"copied to {DRIVE_FOLDER}/reading_model.zip")

from google.colab import files
files.download('/content/reading_model.zip')

## What next

Bring back the three numbers this printed — accuracy, REAL recall, FAKE recall.

Whether it is worth wiring into Django depends on what they say:

- **REAL recall past about 0.7 and accuracy past 70%** — worth deploying. Unzip into `analyzer/reading_model/` and Step 2 gets swapped over.
- **REAL recall still low** — reading was not the bottleneck. Your real articles all coming off one wire service is, and no model fixes that. That is when new data becomes worth the effort.

Nothing in the Django app changes until the numbers justify it — same rule as last time. There is no point building a deployment path for a model that has not earned one.